#Event Hub consumer

Reads what the producer notebook sent and lands it in bronze.

Event Hubs is not Kafka, but it speaks the Kafka protocol on port 9093 - so Spark talks to it
with the plain `kafka` source. No extra library, nothing to install.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, ArrayType)

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "")
dbutils.widgets.text("eventhub_namespace", "")
dbutils.widgets.text("eventhub_name", "")
dbutils.widgets.text("consumer_group", "")

login          = dbutils.widgets.get("login")
catalog        = dbutils.widgets.get("target_catalog")
secret_scope   = dbutils.widgets.get("secret_scope")
secret_key     = dbutils.widgets.get("secret_key")
namespace      = dbutils.widgets.get("eventhub_namespace")
eventhub_name  = dbutils.widgets.get("eventhub_name")
consumer_group = dbutils.widgets.get("consumer_group")

assert all([login, catalog, secret_scope, secret_key, namespace, eventhub_name, consumer_group])

bronze = f"{login}_bronze"
target = f"{catalog}.{bronze}.eventhub_events_bronze"
ckpt   = f"/Volumes/{catalog}/{bronze}/checkpoints/eventhub_events"
print(f"{namespace}/{eventhub_name} -> {target}")

## 1. Connect


In [0]:
conn = dbutils.secrets.get(secret_scope, secret_key)

jaas = ('kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{conn}";')

kafka_options = {
    "kafka.bootstrap.servers":  f"{namespace}.servicebus.windows.net:9093",
    "kafka.security.protocol":  "SASL_SSL",
    "kafka.sasl.mechanism":     "PLAIN",
    "kafka.sasl.jaas.config":   jaas,
    "kafka.group.id":           consumer_group,
    "subscribe":                eventhub_name,
    "startingOffsets":          "earliest",
    "maxOffsetsPerTrigger":     "5000",
    "failOnDataLoss":           "false",
}

## 2. Parse and land it

The broker hands us bytes plus its own metadata. We keep both: the parsed fields for querying and
`partition`/`offset`/`enqueued_ts` because they're the only way to prove later where a row came
from and whether anything was replayed.

No UDF here. `from_json` is a built-in that runs inside the JVM; a Python UDF would serialise every
single row over to a Python process and back, for parsing that Spark already does natively.

In [0]:
EVENT_SCHEMA = StructType([
    StructField("event_id",  StringType()),
    StructField("timestamp", StringType()),
    StructField("camera_id", StringType()),
    StructField("cam_id",    StringType()),
    StructField("producer",  StringType()),
    StructField("sent_ts",   StringType()),
    StructField("payload", StringType()),
])

stream = (spark.readStream.format("kafka")
          .options(**kafka_options)
          .load()
          .select(
              F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"),
              F.col("partition").alias("eh_partition"),
              F.col("offset").alias("eh_offset"),
              F.col("timestamp").alias("enqueued_ts"),
          )
          .select("e.*", "eh_partition", "eh_offset", "enqueued_ts")
          .filter(F.col("producer") == login)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("load_date",    F.current_date()))

q = (stream.writeStream
     .option("checkpointLocation", ckpt)
     .option("mergeSchema", "true")
     .trigger(availableNow=True)
     .toTable(target))
q.awaitTermination()

print(target, "->", spark.table(target).count(), "rows")